# GitHub statistics

Shows the number of newly created repositories and the number of commits per month for the authenticated user.

Set your token before starting Jupyter: `export GITHUB_TOKEN=<your token>` (or enter it when prompted).

In [ ]:
import os, time, getpass
import yaml
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

METADATA_FILE = 'github_stats_metadata.yml'
API = 'https://api.github.com'
session = requests.Session()

def get(url, **params):
    while True:
        r = session.get(url, params=params)
        if r.status_code in (403, 429) and r.headers.get('X-RateLimit-Remaining') == '0' or r.status_code == 429:
            reset = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
            time.sleep(max(reset - time.time(), 1) + 1)
            continue
        r.raise_for_status()
        return r


def fetch_metadata():
    token = os.environ.get('GITHUB_TOKEN') or getpass.getpass('GitHub API token: ')
    session.headers.update({'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'})
    user = get(f'{API}/user').json()['login']

    repos, page = [], 1
    while True:
        batch = get(f'{API}/user/repos', affiliation='owner', per_page=100, page=page).json()
        if not batch:
            break
        repos += batch
        page += 1
    repos = [{k: r.get(k) for k in ('full_name', 'private', 'fork', 'language', 'created_at', 'pushed_at')} for r in repos]

    first = pd.to_datetime([r['created_at'] for r in repos]).min()
    commits = {}
    for start in pd.date_range(first.strftime('%Y-%m-01'), pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d'), freq='MS'):
        end = (start + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
        q = f'author:{user} author-date:{start:%Y-%m-%d}..{end}'
        commits[f'{start:%Y-%m}'] = get(f'{API}/search/commits', q=q, per_page=1).json()['total_count']
        time.sleep(2.1)

    return {'fetched_at': pd.Timestamp.now(tz='UTC').isoformat(), 'user': user,
            'repositories': repos, 'commits_per_month': commits}


def load_metadata():
    """Reuse the yml file if it is less than one month old, otherwise download and store it again."""
    if os.path.exists(METADATA_FILE):
        with open(METADATA_FILE) as f:
            data = yaml.safe_load(f)
        if pd.Timestamp(data['fetched_at']) + pd.DateOffset(months=1) > pd.Timestamp.now(tz='UTC'):
            print('Using cached', METADATA_FILE)
            return data
    data = fetch_metadata()
    with open(METADATA_FILE, 'w') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return data


def plot_monthly(series, title, ylabel):
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.barplot(x=list(range(len(series))), y=series.values, color=sns.color_palette()[0], ax=ax)
    ticks = [i for i, d in enumerate(series.index) if d.month == 1]
    ax.set_xticks(ticks)
    ax.set_xticklabels([series.index[i].year for i in ticks])
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

## Download metadata

The metadata is stored in `github_stats_metadata.yml` and only downloaded again if that file is older than one month. Delete the file to force a refresh. The commit search API is limited to 30 requests per minute, so a fresh download may take a while.

In [ ]:
meta = load_metadata()
print('User:', meta['user'])

## Repositories created per month

In [ ]:
created = pd.to_datetime([r['created_at'] for r in meta['repositories']])
repos_per_month = pd.Series(1, index=created).resample('MS').sum()
repos_per_month = repos_per_month.reindex(
    pd.date_range(repos_per_month.index.min(), pd.Timestamp.now(tz='UTC').normalize().replace(day=1), freq='MS'), fill_value=0)

plot_monthly(repos_per_month, 'New repositories per month', 'repositories')
print('Total repositories:', len(meta['repositories']))

## Commits per month

Commit search covers default branches only.

In [ ]:
commits_per_month = pd.Series(meta['commits_per_month'])
commits_per_month.index = pd.to_datetime(commits_per_month.index).tz_localize('UTC')

plot_monthly(commits_per_month, 'Commits per month', 'commits')
print('Total commits:', commits_per_month.sum())